# Semaine 2 & 3 — Fine-tuning SFT (LoRA) et Alignement DPO

**Objectif** : Spécialiser le modèle Qwen3-1.7B-Base sur notre corpus médical via :
1. **SFT (Supervised Fine-Tuning)** avec LoRA — adaptation efficace sur les 5 000 paires instruction-réponse
2. **DPO (Direct Preference Optimization)** — alignement sur les préférences cliniques

**Prérequis** : Avoir exécuté le notebook `01_data_preparation.ipynb` (datasets dans `data/sft/` et `data/dpo/`)

In [ ]:
# --- RunPod / Jupyter : installation des dépendances (à exécuter UNE fois) ---
# Image ciblée : runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04
# torch est PIN à 2.4.0+cu124 pour empêcher bitsandbytes/trl/etc de le sur-upgrader
# vers une version CUDA 13 incompatible avec le driver RunPod.

%pip install -q --root-user-action=ignore --upgrade pip

%pip install -q --root-user-action=ignore \
    --extra-index-url https://download.pytorch.org/whl/cu124 \
    torch==2.4.0+cu124 torchvision==0.19.0+cu124 torchaudio==2.4.0+cu124 \
    "transformers>=4.47,<5" \
    "accelerate>=1.2,<2" \
    "peft>=0.14,<0.16" \
    "trl>=0.13,<0.16" \
    "bitsandbytes>=0.45,<0.46" \
    "datasets>=3.2,<4" \
    "huggingface_hub>=0.26,<1" \
    "evaluate>=0.4" \
    "scikit-learn>=1.6" \
    "wandb>=0.19" \
    "pyyaml>=6.0" \
    "pandas>=2.2" \
    "tqdm>=4.67" \
    "matplotlib>=3.10" \
    "typing_extensions>=4.14"

# Flash-Attention 2 (compile contre le torch déjà installé — nécessite l'image *-devel*).
# ~5-10 min. Commenter si GPU T4/V100 — et remplacer `flash_attention_2` par `sdpa` dans les YAML.
%pip install -q --root-user-action=ignore flash-attn --no-build-isolation

import torch
print(f"\ntorch : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version : {torch.version.cuda}")
print("\nInstallation terminée. Redémarrer le kernel (Kernel -> Restart) avant de continuer.")

## Authentification Hugging Face

Le dataset `Marintosti/chsa-medical-data` est **privé**. Colle ton token HF ci-dessous (ou définis la variable d'environnement `HF_TOKEN` avant de lancer le kernel).

Génère un token avec le scope **read** sur https://huggingface.co/settings/tokens.

In [ ]:
import os

# Colle ton token ici (ne pas commit !). Laisse tel quel si HF_TOKEN est déjà défini dans l'env.
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "hf_xxxxxxxxxxxxxxxxxxxxxxxxxx")

assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN invalide — colle ton token depuis huggingface.co/settings/tokens"
print("Token HF OK")

In [ ]:
import json
import logging
import os
from pathlib import Path

import torch
import yaml
from datasets import load_dataset
from huggingface_hub import hf_hub_download, login
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd()  # sur RunPod : le notebook est seul, tout se passe dans son dossier courant
print(f"Racine du projet : {PROJECT_ROOT}")
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} Go")

# --- Authentification Hugging Face ---
# Le dataset `Marintosti/chsa-medical-data` est privé : un token HF est requis.
# Recommandé : définir la variable d'environnement HF_TOKEN avant de lancer le notebook
# (ex: `export HF_TOKEN=hf_xxx`, ou "Secrets" sur Kaggle/Colab).
# Dernier recours : `os.environ["HF_TOKEN"] = "hf_..."` dans une cellule (ne pas commit !).
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_DATASET = "Marintosti/chsa-medical-data"

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"Authentification Hugging Face OK — dataset : {HF_DATASET}")
else:
    print("HF_TOKEN non défini : exécuter `os.environ[\'HF_TOKEN\'] = \'hf_...\'` "
          "puis relancer cette cellule avant de charger le dataset privé.")

In [ ]:
# Chargement de la configuration SFT depuis Hugging Face
sft_config_path = hf_hub_download(
    repo_id=HF_DATASET,
    filename="configs/sft_config.yaml",
    repo_type="dataset",
    token=HF_TOKEN,
)
with open(sft_config_path) as f:
    sft_config = yaml.safe_load(f)

print("=== Configuration SFT ===")
print(f"Modèle : {sft_config['model']['name']}")
print(f"LoRA r={sft_config['lora']['r']}, alpha={sft_config['lora']['lora_alpha']}")
print(f"Epochs : {sft_config['training']['num_train_epochs']}")
print(f"Batch size : {sft_config['training']['per_device_train_batch_size']}")
print(f"Gradient accumulation : {sft_config['training']['gradient_accumulation_steps']}")
print(f"Learning rate : {sft_config['training']['learning_rate']}")
print(f"Max seq length : {sft_config['training']['max_seq_length']}")

---
## Partie 1 : Fine-tuning supervisé (SFT) avec LoRA

### 1.1 Vérification des données d'entraînement

In [ ]:
# Chargement des datasets SFT depuis Hugging Face (repo: Marintosti/chsa-medical-data)
sft_train = load_dataset(HF_DATASET, data_files="sft/train.jsonl", split="train", token=HF_TOKEN)
sft_eval = load_dataset(HF_DATASET, data_files="sft/eval.jsonl", split="train", token=HF_TOKEN)

print(f"SFT Train : {len(sft_train)} exemples")
print(f"SFT Eval  : {len(sft_eval)} exemples")
print(f"\nColonnes : {sft_train.column_names}")
print(f"\nExemple (premier message user) :")
print(sft_train[0]["messages"][1]["content"][:300])

In [ ]:
# Distribution des sources et langues dans le train set
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

sources = Counter(sft_train["source"])
langues = Counter(sft_train["langue"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Par source
labels = [s.split("/")[-1] for s in sources.keys()]
axes[0].bar(labels, sources.values(), color=plt.cm.Set2(np.linspace(0, 1, len(sources))))
axes[0].set_title("Distribution par source (SFT train)")
axes[0].set_ylabel("Nombre d'exemples")
for i, v in enumerate(sources.values()):
    axes[0].text(i, v + 20, str(v), ha="center", fontsize=9)

# Par langue
axes[1].pie(langues.values(), labels=langues.keys(), autopct="%1.0f%%",
            colors=["#66b3ff", "#ff9999"], startangle=90)
axes[1].set_title("Distribution par langue (SFT train)")

plt.tight_layout()
plt.show()

### 1.2 Chargement du modèle de base et configuration LoRA

In [ ]:
# Chargement du modèle Qwen3-1.7B-Base
model_name = sft_config["model"]["name"]
dtype = getattr(torch, sft_config["model"]["torch_dtype"])

print(f"Chargement de {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    attn_implementation=sft_config["model"].get("attn_implementation"),
    device_map="auto",
    trust_remote_code=True,
)

print(f"Modèle chargé : {model_name}")
print(f"Paramètres totaux : {model.num_parameters():,}")
print(f"dtype : {dtype}")

In [ ]:
# Configuration LoRA
lora_cfg = sft_config["lora"]

peft_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    modules_to_save=lora_cfg.get("modules_to_save"),  # embeddings + lm_head si spécifiés
    task_type=TaskType.CAUSAL_LM,
)

print("=== Configuration LoRA ===")
print(f"Rang (r) : {peft_config.r}")
print(f"Alpha : {peft_config.lora_alpha}")
print(f"Dropout : {peft_config.lora_dropout}")
print(f"Modules ciblés (LoRA) : {peft_config.target_modules}")
print(f"Modules entraînés FULL : {peft_config.modules_to_save}")
print(f"\nRatio alpha/r = {peft_config.lora_alpha / peft_config.r:.1f} (scaling factor)")

### 1.3 Test du modèle de base (avant fine-tuning)

On teste le modèle de base sur quelques questions médicales pour avoir un point de comparaison.

In [ ]:
# Inférence de base — avant SFT
test_prompts = [
    "Un patient de 55 ans se présente aux urgences avec une douleur thoracique aiguë irradiant vers le bras gauche. Quel est le niveau de priorité ?",
    "Une femme de 30 ans consulte pour un mal de gorge depuis 2 jours avec une légère fièvre à 38.2°C. Quel est le niveau de priorité ?",
    "Un enfant de 3 ans est amené pour une éruption cutanée diffuse apparue il y a 1 heure après avoir mangé des cacahuètes. Il présente un gonflement des lèvres. Quel est le niveau de priorité ?",
]

print("=== Réponses du modèle de BASE (avant SFT) ===")
print("=" * 60)

model.eval()
base_responses = []
for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "Vous etes un assistant medical specialise dans le triage des urgences."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=1.0,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    base_responses.append(response)
    print(f"\n--- Cas {i+1} ---")
    print(f"Q: {prompt[:100]}...")
    print(f"R: {response[:500]}")
    print()

### 1.4 Lancement de l'entraînement SFT

In [ ]:
# Configuration du SFTTrainer
training_cfg = sft_config["training"]

sft_training_args = SFTConfig(
    output_dir=str(PROJECT_ROOT / training_cfg["output_dir"]),
    num_train_epochs=training_cfg["num_train_epochs"],
    per_device_train_batch_size=training_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=training_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=training_cfg["gradient_accumulation_steps"],
    learning_rate=training_cfg["learning_rate"],
    weight_decay=training_cfg["weight_decay"],
    warmup_ratio=training_cfg["warmup_ratio"],
    lr_scheduler_type=training_cfg["lr_scheduler_type"],
    logging_steps=training_cfg["logging_steps"],
    eval_strategy=training_cfg["eval_strategy"],
    eval_steps=training_cfg["eval_steps"],
    save_strategy=training_cfg["save_strategy"],
    save_steps=training_cfg["save_steps"],
    save_total_limit=training_cfg["save_total_limit"],
    bf16=training_cfg["bf16"],
    max_seq_length=training_cfg["max_seq_length"],
    gradient_checkpointing=training_cfg["gradient_checkpointing"],
    report_to=training_cfg["report_to"],
    seed=training_cfg.get("seed", 42),
)

# Le SFTTrainer gère l'application de LoRA via peft_config
trainer = SFTTrainer(
    model=model,
    args=sft_training_args,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    processing_class=tokenizer,
    peft_config=peft_config,
)

# Afficher les paramètres entraînables
trainer.model.print_trainable_parameters()

In [ ]:
# Lancement de l'entraînement SFT
print("Démarrage de l'entraînement SFT...")
sft_results = trainer.train()
print(f"\nEntraînement terminé !")
print(f"Loss finale : {sft_results.training_loss:.4f}")

In [ ]:
# Sauvegarde du modèle SFT final
sft_output_dir = PROJECT_ROOT / training_cfg["output_dir"] / "final"
trainer.save_model(str(sft_output_dir))
tokenizer.save_pretrained(str(sft_output_dir))
print(f"Modèle SFT sauvegardé dans : {sft_output_dir}")

### 1.5 Visualisation des métriques SFT

In [ ]:
# Courbes d'entraînement SFT
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

# Extraire les métriques
train_losses = [(h["step"], h["loss"]) for h in log_history if "loss" in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval loss", marker="o", markersize=4)

ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.set_title("SFT — Courbes de loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if eval_losses:
    print(f"Meilleure eval loss : {min(l for _, l in eval_losses):.4f}")

### 1.6 Test du modèle SFT (après fine-tuning)

In [ ]:
# Inférence après SFT — sampling + repetition_penalty pour casser les boucles
# Qwen3-Base n'a jamais vu du ChatML → les premiers tokens sont instables avec greedy decoding.
# Le sampling + repetition_penalty corrige ces deux problèmes sans retraining.
print("=== Réponses du modèle SFT (après fine-tuning, sampling) ===")
print("=" * 60)

sft_model = trainer.model
sft_model.eval()

sft_responses = []
for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "Vous etes un assistant medical specialise dans le triage des urgences."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(sft_model.device)

    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    sft_responses.append(response)
    print(f"\n--- Cas {i+1} ---")
    print(f"Q: {prompt[:100]}...")
    print(f"R: {response[:500]}")
    print()

In [ ]:
# Comparaison côte à côte : Base vs SFT
print("=" * 80)
print("COMPARAISON : Modèle de base vs Modèle SFT")
print("=" * 80)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 80}")
    print(f"CAS {i+1}: {prompt[:100]}...")
    print(f"{'─' * 80}")
    print(f"\n[BASE] {base_responses[i][:300]}")
    print(f"\n[SFT]  {sft_responses[i][:300]}")

### 🔍 Diagnostic SFT (réponses dégradées)

Les deux cellules suivantes permettent d'identifier si le problème vient des **données d'entraînement** (présence de placeholders Presidio comme `<PERSONNE>`) ou des **paramètres d'inférence** (greedy → boucles).

In [ ]:
# DIAGNOSTIC 1 : inspection des données SFT
import re

print("=== Exemple 0 ===")
for m in sft_train[0]["messages"]:
    print(f"[{m['role']}] {m['content'][:400]}")

print("\n=== Exemple 100 ===")
for m in sft_train[100]["messages"]:
    print(f"[{m['role']}] {m['content'][:400]}")

# Compte les placeholders type <PERSONNE>, <DATE>, etc.
placeholder_counts = {}
for ex in sft_train:
    for m in ex["messages"]:
        for tag in re.findall(r"<[A-Z_]+>", m["content"]):
            placeholder_counts[tag] = placeholder_counts.get(tag, 0) + 1

print("\n=== Placeholders détectés dans le train set ===")
if placeholder_counts:
    for tag, n in sorted(placeholder_counts.items(), key=lambda x: -x[1]):
        print(f"  {tag}: {n}")
else:
    print("  (aucun — les données ne sont pas anonymisées, le bug vient d'ailleurs)")

In [ ]:
# DIAGNOSTIC 2 : inférence avec sampling + repetition_penalty
# (si c'était juste un problème de boucle greedy, ça donnera un résultat lisible)
import torch

messages = [
    {"role": "system", "content": "Vous etes un assistant medical specialise dans le triage des urgences."},
    {"role": "user", "content": test_prompts[0]},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("=== PROMPT ENVOYÉ AU MODÈLE ===")
print(text)
print("=" * 60)

inputs = tokenizer(text, return_tensors="pt").to(sft_model.device)
with torch.no_grad():
    out = sft_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
    )
print("\n=== RÉPONSE SFT (sampling + repetition penalty) ===")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
# DIAGNOSTIC : chat template + premier exemple tokenisé
# Permet de voir EXACTEMENT ce que le modèle a appris pendant le SFT.
print("=== CHAT TEMPLATE DU TOKENIZER ===")
template = tokenizer.chat_template or "(aucun — template par défaut)"
print(template[:800])
print()

print("=== TOKENS SPÉCIAUX ===")
print(f"  BOS: {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")
print(f"  EOS: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  PAD: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print()

print("=== EXEMPLE TRAINING 0 APRÈS APPLY_CHAT_TEMPLATE ===")
example_text = tokenizer.apply_chat_template(
    sft_train[0]["messages"], tokenize=False, add_generation_prompt=False
)
print(example_text[:800])
print()

print("=== PREMIERS TOKENS DE LA RÉPONSE ASSISTANT ===")
# Isole la partie "assistant" pour voir comment elle est tokenisée
assistant_start = example_text.find("assistant")
if assistant_start != -1:
    assistant_part = example_text[assistant_start:assistant_start+200]
    ids = tokenizer.encode(assistant_part, add_special_tokens=False)
    print(f"Texte : {assistant_part[:150]}")
    print(f"IDs (20 premiers) : {ids[:20]}")
    print(f"Tokens (20 premiers) : {[tokenizer.decode([i]) for i in ids[:20]]}")

---
## Partie 2 : Alignement par préférences (DPO)

On prend le modèle SFT fine-tuné et on l'aligne avec DPO sur les paires chosen/rejected du dataset UltraMedical-Preference.

### 2.1 Chargement de la configuration DPO

In [ ]:
# Libérer la mémoire GPU du SFT
del trainer, sft_model, model
torch.cuda.empty_cache()

from trl import DPOConfig, DPOTrainer

# Chargement de la config DPO depuis Hugging Face
dpo_config_path = hf_hub_download(
    repo_id=HF_DATASET,
    filename="configs/dpo_config.yaml",
    repo_type="dataset",
    token=HF_TOKEN,
)
with open(dpo_config_path) as f:
    dpo_config = yaml.safe_load(f)

print("=== Configuration DPO ===")
print(f"Modèle SFT : {dpo_config['model']['name']}")
print(f"Beta (DPO) : {dpo_config['training']['beta']}")
print(f"Learning rate : {dpo_config['training']['learning_rate']}")
print(f"Epochs : {dpo_config['training']['num_train_epochs']}")
print(f"Max length : {dpo_config['training']['max_length']}")

### 2.2 Chargement du modèle SFT et des données DPO

In [ ]:
# Chargement du modèle SFT fine-tuné
sft_model_path = str(PROJECT_ROOT / dpo_config["model"]["name"])
dtype = getattr(torch, dpo_config["model"]["torch_dtype"])

print(f"Chargement du modèle SFT depuis : {sft_model_path}")

tokenizer = AutoTokenizer.from_pretrained(sft_model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dpo_model = AutoModelForCausalLM.from_pretrained(
    sft_model_path,
    torch_dtype=dtype,
    attn_implementation=dpo_config["model"].get("attn_implementation"),
    device_map="auto",
    trust_remote_code=True,
)

print(f"Modèle chargé : {dpo_model.num_parameters():,} paramètres")

In [ ]:
# Chargement des données DPO depuis Hugging Face (repo: Marintosti/chsa-medical-data)
dpo_train = load_dataset(HF_DATASET, data_files="dpo/train.jsonl", split="train", token=HF_TOKEN)
dpo_eval = load_dataset(HF_DATASET, data_files="dpo/eval.jsonl", split="train", token=HF_TOKEN)

# Subsampling optionnel (défini dans dpo_config.yaml pour gagner du temps POC)
max_train = dpo_config["data"].get("max_samples")
max_eval = dpo_config["data"].get("max_eval_samples")
if max_train and max_train < len(dpo_train):
    dpo_train = dpo_train.shuffle(seed=42).select(range(max_train))
    print(f"[subsampling] DPO train : {max_train} exemples sélectionnés")
if max_eval and max_eval < len(dpo_eval):
    dpo_eval = dpo_eval.shuffle(seed=42).select(range(max_eval))
    print(f"[subsampling] DPO eval  : {max_eval} exemples sélectionnés")

print(f"\nDPO Train : {len(dpo_train)} exemples")
print(f"DPO Eval  : {len(dpo_eval)} exemples")
print(f"\nColonnes : {dpo_train.column_names}")
print(f"\nExemple de prompt :")
print(dpo_train[0]["prompt"][:300])

In [ ]:
# Configuration LoRA pour DPO
dpo_lora_cfg = dpo_config["lora"]

dpo_peft_config = LoraConfig(
    r=dpo_lora_cfg["r"],
    lora_alpha=dpo_lora_cfg["lora_alpha"],
    lora_dropout=dpo_lora_cfg["lora_dropout"],
    target_modules=dpo_lora_cfg["target_modules"],
    modules_to_save=dpo_lora_cfg.get("modules_to_save"),
    task_type=TaskType.CAUSAL_LM,
)

### 2.3 Lancement de l'entraînement DPO

In [ ]:
# Configuration du DPOTrainer
dpo_training_cfg = dpo_config["training"]

dpo_training_args = DPOConfig(
    output_dir=str(PROJECT_ROOT / dpo_training_cfg["output_dir"]),
    num_train_epochs=dpo_training_cfg["num_train_epochs"],
    per_device_train_batch_size=dpo_training_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=dpo_training_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=dpo_training_cfg["gradient_accumulation_steps"],
    learning_rate=dpo_training_cfg["learning_rate"],
    weight_decay=dpo_training_cfg["weight_decay"],
    warmup_ratio=dpo_training_cfg["warmup_ratio"],
    lr_scheduler_type=dpo_training_cfg["lr_scheduler_type"],
    logging_steps=dpo_training_cfg["logging_steps"],
    eval_strategy=dpo_training_cfg["eval_strategy"],
    eval_steps=dpo_training_cfg["eval_steps"],
    save_strategy=dpo_training_cfg["save_strategy"],
    save_steps=dpo_training_cfg["save_steps"],
    save_total_limit=dpo_training_cfg["save_total_limit"],
    bf16=dpo_training_cfg["bf16"],
    max_length=dpo_training_cfg["max_length"],
    max_prompt_length=dpo_training_cfg["max_prompt_length"],
    beta=dpo_training_cfg["beta"],
    gradient_checkpointing=dpo_training_cfg["gradient_checkpointing"],
    report_to=dpo_training_cfg["report_to"],
    seed=dpo_training_cfg.get("seed", 42),
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_training_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer,
    peft_config=dpo_peft_config,
)

dpo_trainer.model.print_trainable_parameters()

In [ ]:
# Lancement de l'entraînement DPO
print("Démarrage de l'entraînement DPO...")
dpo_results = dpo_trainer.train()
print(f"\nEntraînement DPO terminé !")
print(f"Loss finale : {dpo_results.training_loss:.4f}")

In [ ]:
# Sauvegarde du modèle DPO final
dpo_output_dir = PROJECT_ROOT / dpo_training_cfg["output_dir"] / "final"
dpo_trainer.save_model(str(dpo_output_dir))
tokenizer.save_pretrained(str(dpo_output_dir))
print(f"Modèle DPO sauvegardé dans : {dpo_output_dir}")

### 2.4 Visualisation des métriques DPO

In [ ]:
# Courbes d'entraînement DPO
import matplotlib.pyplot as plt

dpo_log_history = dpo_trainer.state.log_history

# Extraire les métriques DPO
train_losses = [(h["step"], h["loss"]) for h in dpo_log_history if "loss" in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in dpo_log_history if "eval_loss" in h]
rewards_chosen = [(h["step"], h["rewards/chosen"]) for h in dpo_log_history if "rewards/chosen" in h]
rewards_rejected = [(h["step"], h["rewards/rejected"]) for h in dpo_log_history if "rewards/rejected" in h]
reward_margins = [(h["step"], h["rewards/margins"]) for h in dpo_log_history if "rewards/margins" in h]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, label="Train loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    axes[0].plot(steps, losses, label="Eval loss", marker="o", markersize=4)
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Loss")
axes[0].set_title("DPO — Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rewards
if rewards_chosen:
    steps, rewards = zip(*rewards_chosen)
    axes[1].plot(steps, rewards, label="Chosen", color="green")
if rewards_rejected:
    steps, rewards = zip(*rewards_rejected)
    axes[1].plot(steps, rewards, label="Rejected", color="red")
axes[1].set_xlabel("Steps")
axes[1].set_ylabel("Reward")
axes[1].set_title("DPO — Rewards (chosen vs rejected)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Reward margin
if reward_margins:
    steps, margins = zip(*reward_margins)
    axes[2].plot(steps, margins, label="Margin", color="purple")
    axes[2].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[2].set_xlabel("Steps")
axes[2].set_ylabel("Margin")
axes[2].set_title("DPO — Reward margin (chosen - rejected)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5 Test du modèle DPO (après alignement)

In [ ]:
# Inférence après DPO — même prompts
print("=== Réponses du modèle DPO (après alignement) ===")
print("=" * 60)

dpo_final_model = dpo_trainer.model
dpo_final_model.eval()

dpo_responses = []
for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "Vous etes un assistant medical specialise dans le triage des urgences."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(dpo_final_model.device)

    with torch.no_grad():
        outputs = dpo_final_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=1.0,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    dpo_responses.append(response)
    print(f"\n--- Cas {i+1} ---")
    print(f"Q: {prompt[:100]}...")
    print(f"R: {response[:500]}")
    print()

---
## Partie 3 : Comparaison finale Base vs SFT vs DPO

In [ ]:
# Tableau comparatif
print("=" * 100)
print("COMPARAISON FINALE : Base vs SFT vs DPO")
print("=" * 100)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'━' * 100}")
    print(f"CAS {i+1}: {prompt}")
    print(f"{'━' * 100}")
    print(f"\n🔵 [BASE]")
    print(base_responses[i][:400])
    print(f"\n🟢 [SFT]")
    print(sft_responses[i][:400])
    print(f"\n🟣 [DPO]")
    print(dpo_responses[i][:400])
    print()

---
## Résumé

| Phase | Modèle | Description |
|-------|--------|-------------|
| Base | Qwen3-1.7B-Base | Modèle pré-entraîné, pas de spécialisation médicale |
| SFT | + LoRA fine-tuning | Spécialisé sur 5 000 paires médicales (FR/EN) |
| DPO | + Alignement préférences | Aligné sur les pratiques cliniques validées |

**Fichiers produits :**
- `outputs/sft/final/` — Modèle SFT (adaptateurs LoRA + tokenizer)
- `outputs/dpo/final/` — Modèle DPO final (adaptateurs LoRA + tokenizer)
- Checkpoints intermédiaires dans `outputs/sft/` et `outputs/dpo/`

**Prochaine étape :** Évaluation complète et déploiement (Semaine 4)